# 02. Limpieza de datos

**Objetivo**: dejar las 7 tablas en un estado en el que se pueda confiar en ellas — fechas y textos estandarizados, duplicados resueltos, nulos decididos (rellenados, documentados como estructurales, o dejados como están con justificación) — sin calcular todavía ninguna métrica.

In [1]:
# Cargamos las librerias
import pandas as pd

In [2]:
# Lista con los nombres base de los 7 archivos 
ficheros = ['usuarios','sesiones','partidas','compras','ingresos_ads','gasto_ua','productos_iap']

# Carpeta donde están los CSV, 
ruta = '../data/raw/'

# Diccionario vacio para guardar los DataFrames
dfs = {}

# Bucle para recorrer la lista y construir la ruta completa
for f in ficheros:
    dfs[f] = pd.read_csv(f"{ruta}{f}.csv")
    # Imprimimos el nombre del fichero y su forma (filas, columnas).
    print(f"{f}: {dfs[f].shape}")


usuarios: (8040, 6)
sesiones: (44945, 5)
partidas: (61716, 7)
compras: (4558, 8)
ingresos_ads: (19999, 6)
gasto_ua: (280, 6)
productos_iap: (14, 5)


### Limpieza `usuarios.csv`

In [3]:
# Asignamos el dataframe a una variable
usuarios_limpio = dfs['usuarios'].copy()

print(usuarios_limpio.dtypes)

user_id              int64
fecha_instalacion      str
canal_adquisicion      str
pais                   str
plataforma             str
version_app            str
dtype: object


In [4]:
# Empezamos con la fecha, habia varios formatos 
print(usuarios_limpio['fecha_instalacion'].unique())

<StringArray>
[ '9 jul 2024',  '12/07/2024',  '2024-06-19',  '2024-07-03',  '2024-07-01',
  '03/07/2024',  '28/06/2024',  '2024-07-04',  '15/06/2024',  '09/06/2024',
 ...
 '12 jul 2024',  '2024-05-31', '28 may 2024', '11 jul 2024', '20 jun 2024',
 '29 jun 2024',  '5 jul 2024', '29 may 2024', '11 jun 2024', '13 jun 2024']
Length: 165, dtype: str


In [5]:
# Cambiamos a formato fecha los diferrentes tipos 
usuarios_limpio['fecha_instalacion'] = pd.to_datetime(usuarios_limpio['fecha_instalacion'],
                                                       format='mixed', dayfirst=True, errors='coerce')

In [6]:
# Al poner coerce si algun valor no se interpreta el programa lo convierte en NaT, nos aseguramos de que no genere nulos
print(f"Fechas no reconocidas (NaT): {usuarios_limpio['fecha_instalacion'].isnull().sum()}")
print(f"Primera instalacion: {usuarios_limpio['fecha_instalacion'].min().date()}")
print(f"Ultima instalacion: {usuarios_limpio['fecha_instalacion'].max().date()}")

Fechas no reconocidas (NaT): 0
Primera instalacion: 2024-01-06
Ultima instalacion: 2024-12-07


In [7]:
# Comprobamos los diferentes valores de canal_adquisicion
print(f"Grafias antes: {usuarios_limpio['canal_adquisicion'].nunique()}")
for canal,n in usuarios_limpio['canal_adquisicion'].value_counts().items():
    print(f"{canal}: {n}")

Grafias antes: 23
coinrush: 1281
CoinRush: 855
meta: 636
google: 578
Meta: 409
coin_rush: 407
Google Ads: 399
facebook: 397
tiktok: 355
apple_search: 311
google_ads: 279
ASA: 279
Coinrush: 270
organico: 266
TikTok: 223
Apple Search: 179
Facebook: 168
orgánico: 159
Google: 151
apple search: 137
organic: 132
tik_tok: 124
Organico: 45


In [8]:
# Quitamos mayuculas, espacios y mapeamos para formatear los diferentes valors
mapa = {
    'coinrush': 'coinrush', 'coin_rush': 'coinrush',
    'meta': 'meta', 'facebook': 'meta',
    'google': 'google', 'google ads': 'google', 'google_ads': 'google',
    'apple_search': 'apple_search', 'asa': 'apple_search', 'apple search': 'apple_search',
    'tiktok': 'tiktok', 'tik_tok': 'tiktok',
    'organico': 'organico', 'orgánico': 'organico', 'organic': 'organico',
}

usuarios_limpio['canal_adquisicion'] =  usuarios_limpio['canal_adquisicion'].str.lower().str.strip().map(mapa)

print(f"Grafias despues: {usuarios_limpio['canal_adquisicion'].nunique()}")
for canal,n in usuarios_limpio['canal_adquisicion'].value_counts().items():
    print(f"{canal}: {n}")

Grafias despues: 6
coinrush: 2813
meta: 1610
google: 1407
apple_search: 906
tiktok: 702
organico: 602


In [9]:
# Pasamos al pais, comprobamos primero los valores para sacar el mapa 
print(f"Grafias antes: {usuarios_limpio['pais'].nunique()}")
for pais,n in usuarios_limpio['pais'].value_counts().items():
    print(f"{pais}: {n}")

Grafias antes: 12
ES: 1864
MX: 1268
España: 834
PT: 763
es: 680
IT: 568
México: 554
mexico: 459
Portugal: 348
pt: 292
Italia: 244
it: 166


In [10]:
# Estandarizamos formatos
mapa_pais = {
    'es': 'ES', 'españa': 'ES',
    'mx': 'MX', 'méxico': 'MX', 'mexico': 'MX',
    'pt': 'PT', 'portugal': 'PT',
    'it': 'IT', 'italia': 'IT',
}
usuarios_limpio['pais'] = usuarios_limpio['pais'].str.lower().str.strip().map(mapa_pais)

# Comprobación de nulos
print(f"Nulos tras mapear pais: {usuarios_limpio['pais'].isnull().sum()}")

print(f"Grafias despues: {usuarios_limpio['pais'].nunique()}")
for pais, n in usuarios_limpio['pais'].value_counts().items():
    print(f"{pais}: {n}")

Nulos tras mapear pais: 0
Grafias despues: 4
ES: 3378
MX: 2281
PT: 1403
IT: 978


In [11]:
# Pasamos a la plataforma:
print(f"Grafias antes: {usuarios_limpio['plataforma'].nunique()}")
for plataforma, n in usuarios_limpio['plataforma'].value_counts().items():
    print(f"{plataforma}: {n}")

Grafias antes: 6
android: 1995
Android: 1747
ios: 1641
iOS: 1485
AND: 629
IOS: 543


In [12]:
# Transformamos estandarizando
mapa_plataforma = {
    'android': 'Android',
    'and': 'Android',
    'ios': 'iOS',
}

usuarios_limpio['plataforma'] = usuarios_limpio['plataforma'].str.lower().str.strip().map(mapa_plataforma)

print(f"Nulos tras la estandarizacion: {usuarios_limpio['plataforma'].isnull().sum()}")
print(f"Grafias despues: {usuarios_limpio['plataforma'].nunique()}")
for plataforma, n in usuarios_limpio['plataforma'].value_counts().items():
    print(f"{plataforma}: {n}")


Nulos tras la estandarizacion: 0
Grafias despues: 2
Android: 4371
iOS: 3669


In [13]:
# Duplicados de user_id
antes = len(usuarios_limpio)

usuarios_limpio = usuarios_limpio.sort_values('fecha_instalacion').drop_duplicates(subset='user_id',keep='last')

print(f"Filas antes:{antes}")
print(f"Filas despues: {len(usuarios_limpio)}")

Filas antes:8040
Filas despues: 8000


### Limpieza `usuarios.csv`

- `fecha_instalacion` → convertida a `datetime` (0 nulos).
- `canal_adquisicion` → estandarizada, 6 canales reales.
- `pais` → estandarizada, 4 países (`ES`, `MX`, `PT`, `IT`).
- `plataforma` → estandarizada (`Android`, `iOS`).
- **Duplicados**: `drop_duplicates()` resolvió los 40 casos sin pérdida de información real (8.040 → 8.000 filas).
- `version_app`: 297 nulos en los datos originales, se dejan como están (no interviene en las métricas del encuadre).

### Limpieza 'sesiones.csv'


In [14]:
sesiones_limpio = dfs['sesiones'].copy()

sesiones_limpio.dtypes

evento_id            int64
user_id              int64
fecha                  str
duracion_segundos    int64
plataforma             str
dtype: object

In [15]:
# Convertimos los textos a tipo datetime; no fijamos un formato de visualización.
sesiones_limpio['fecha'] = pd.to_datetime(sesiones_limpio['fecha'],format='mixed',dayfirst=True, errors='coerce')

print(f"Fechas nulas: {sesiones_limpio['fecha'].isnull().sum()}")

Fechas nulas: 0


In [16]:
# Pasamos a la plataforma
print(f"Grafias antes: {sesiones_limpio['plataforma'].nunique()}")
for plataforma, n in sesiones_limpio['plataforma'].value_counts().items():
    print(f"{plataforma}: {n}")

Grafias antes: 6
android: 10765
Android: 9751
ios: 9451
iOS: 8287
AND: 3627
IOS: 3064


In [17]:
# Estandarizamos las plataformas, es igual que en usuarios
sesiones_limpio['plataforma'] =  sesiones_limpio['plataforma'].str.lower().str.strip().map(mapa_plataforma)

print(f"Nulos tras la estandarizacion: {sesiones_limpio['plataforma'].isnull().sum()}")
print(f"Grafias despues: {sesiones_limpio['plataforma'].nunique()}")

Nulos tras la estandarizacion: 0
Grafias despues: 2


In [18]:
# Seguimos con las sesiones, de duraciones imposibles 
sesiones_antes = len(sesiones_limpio)
print(sesiones_limpio['duracion_segundos'].describe())

malas = ((sesiones_limpio['duracion_segundos'] <= 0) | (sesiones_limpio['duracion_segundos'] > 3600))
 
print("\nSesiones antes: ",sesiones_antes)
print(f"Sesiones fuera de rango: {malas.sum()}")
print(f"Duracion media SIN limpiar: {sesiones_limpio['duracion_segundos'].mean():.1f} s")


count     44945.000000
mean        583.905039
std       20005.028244
min         -30.000000
25%         104.000000
50%         180.000000
75%         255.000000
max      999999.000000
Name: duracion_segundos, dtype: float64

Sesiones antes:  44945
Sesiones fuera de rango: 60
Duracion media SIN limpiar: 583.9 s


In [19]:
sesiones_limpio = sesiones_limpio[sesiones_limpio['duracion_segundos'].between(1, 3600)]

print("\nSesiones despues: ",len(sesiones_limpio))
print(f"Duracion media limpia: {sesiones_limpio['duracion_segundos'].mean():.1f} s")



Sesiones despues:  44885
Duracion media limpia: 183.7 s


In [20]:
# Eliminamos los 312 duplicados que habia exactos
print(f"Duplicados exactos: {sesiones_limpio.duplicated().sum()}")

sesiones_limpio = sesiones_limpio.drop_duplicates()

print(f"Filas despues: {len(sesiones_limpio)}")
print(f"Duplicados ahora: {sesiones_limpio.duplicated().sum()}")

Duplicados exactos: 312
Filas despues: 44573
Duplicados ahora: 0


### Limpieza de `sesiones.csv`

- **Filas originales:** 44,945
- **`fecha`**: convertida a datetime (formatos mixtos, sin nulos resultantes)
- **`plataforma`**: estandarizada a `Android` / `iOS` (6 variantes de texto unificadas)
- **`duracion_segundos`**: eliminadas 60 filas fuera de rango (negativos + valor sentinela 999999), aplicando criterio de negocio (máximo 1 hora = 3600s). Media pasó de 583.9s (deformada por outliers) a 183.6s, coherente con la mediana (180s).
- **Duplicados exactos**: eliminadas 312 filas (evento_id repetido con fila idéntica)
- **Filas finales:** 44,573

### Limpieza `partidas.csv`

In [21]:
partidas_limpio = dfs['partidas'].copy()

print(partidas_limpio.dtypes)

partida_id           int64
user_id              int64
nivel                int64
fecha                  str
resultado              str
movimientos          int64
boosters_usados    float64
dtype: object


In [22]:
# Limpiamos la fecha
partidas_limpio['fecha'] = pd.to_datetime(partidas_limpio['fecha'],format='mixed',dayfirst=True,errors='coerce')

print(partidas_limpio['fecha'].dtype)
print(f"Valores nulos o sin reconocer:",partidas_limpio['fecha'].isnull().sum())

datetime64[us]
Valores nulos o sin reconocer: 0


In [23]:
# Comprobamos los valores de resultado para la normalizacion
print("Tipos de resultado")
print(partidas_limpio['resultado'].value_counts())

Tipos de resultado
resultado
win        11733
WIN        11728
ganada     11624
fail        8916
lose        8884
perdida     8831
Name: count, dtype: int64


In [24]:
# Estandarizamos los formatos a 1 (win) y 0 (lose)
mapa_resultado = {
    'win': 1,
    'ganada': 1,
    'fail': 0,
    'lose': 0,
    'perdida': 0,
}

partidas_limpio['resultado'] = partidas_limpio['resultado'].str.lower().str.strip().map(mapa_resultado)

print("Tipos de resultado: ",partidas_limpio['resultado'].value_counts())

Tipos de resultado:  resultado
1    35085
0    26631
Name: count, dtype: int64


In [25]:
# Contamos las partidas para cada cantidad de boosters usados
print("Distribucion de boosters")
print(partidas_limpio['boosters_usados'].value_counts(dropna=False).sort_index())

Distribucion de boosters
boosters_usados
0.0    37020
1.0    14424
2.0     5966
3.0     2452
NaN     1854
Name: count, dtype: int64


In [26]:
# Creamos una columna auxiliar que indica si falta el dato de boosters.
revision_boosters =  partidas_limpio.assign(booster_nulo=partidas_limpio['boosters_usados'].isna())

# Reunimos las filas que pertenecen al mismo nivel y calculamos dos recuentos para cada grupo
resumen_boosters = revision_boosters.groupby('nivel').agg(
    total_partidas=('booster_nulo','size'),
    nulos_boosters=('booster_nulo','sum'))

# Para cada nivel, dividimos las partidas con dato vacío entre el total de ese nivel.
resumen_boosters['porcentajes_nulos'] = (resumen_boosters['nulos_boosters']
                                      / resumen_boosters['total_partidas'] * 100)

print(resumen_boosters.sort_values('porcentajes_nulos',ascending=False))

       total_partidas  nulos_boosters  porcentajes_nulos
nivel                                                   
15                203              10           4.926108
7                2438              82           3.363413
2                9074             287           3.162883
9                1541              48           3.114860
4                6880             214           3.110465
6               11431             352           3.079346
3                8019             245           3.055244
8                1942              58           2.986612
13                503              15           2.982107
1               10214             295           2.888193
10               1489              43           2.887844
5                5800             162           2.793103
14                356               9           2.528090
12                908              18           1.982379
11                918              16           1.742919


In [27]:
# Contamos las filas que son idénticas en todas sus columnas y las que se duplica el id
duplicados_exactos = partidas_limpio.duplicated().sum()
duplicados_id =  partidas_limpio.duplicated(subset=['partida_id']).sum()

# Mostramos ambos recuentos antes de decidir si hay que eliminar algo.
print(f"Filas duplicadas exactas: {duplicados_exactos}")
print(f"Repeticiones de partida_id: {duplicados_id}")


Filas duplicadas exactas: 307
Repeticiones de partida_id: 307


In [28]:
filas_antes = len(partidas_limpio)

# Eliminamos las filas idénticas en todas sus columnas.
partidas_limpio = partidas_limpio.drop_duplicates()

# Comprobamos cuántas filas hemos eliminado y cuántas quedan.
print(f"Filas eliminadas: {filas_antes - len(partidas_limpio)}")
print(f"Partidas restantes: {len(partidas_limpio)}")

# Recalculamos las repeticiones sobre la tabla actual; el recuento anterior no se actualiza solo.
print(
    "Repeticiones de partida_id tras la limpieza:",
    partidas_limpio.duplicated(subset=['partida_id']).sum()
)

Filas eliminadas: 307
Partidas restantes: 61409
Repeticiones de partida_id tras la limpieza: 0


In [29]:
# Resumimos los movimientos: cantidad de datos, media, mínimo,
print("Resumen movimientos:")
print(partidas_limpio['movimientos'].describe())

# Contamos nulos, ceros y negativos
print("\nMovimientos nulos:",partidas_limpio['movimientos'].isna().sum())
print("Movimientos iguales a cero:", (partidas_limpio['movimientos'] == 0).sum())
print("Movimientos negativos:", (partidas_limpio['movimientos'] < 0).sum())

Resumen movimientos:
count    61409.000000
mean        21.577749
std          7.903680
min         -3.000000
25%         16.000000
50%         22.000000
75%         27.000000
max         58.000000
Name: movimientos, dtype: float64

Movimientos nulos: 0
Movimientos iguales a cero: 13
Movimientos negativos: 32


In [30]:
# Partidas con movimientos negativos y 0
partidas_negativas = partidas_limpio.loc[partidas_limpio['movimientos']<0]
partidas_cero = partidas_limpio.loc[partidas_limpio['movimientos']==0]

# Contamos resultados
print("Resultado partidas con movimientos negativos:")
print(partidas_negativas['resultado'].value_counts(dropna=False))
print("\nResultados de las partidas con cero movimientos:")
print(partidas_cero['resultado'].value_counts(dropna=False))

Resultado partidas con movimientos negativos:
resultado
1    18
0    14
Name: count, dtype: int64

Resultados de las partidas con cero movimientos:
resultado
1    11
0     2
Name: count, dtype: int64


In [31]:
# Identificamos las partidas con un numero negativo 
movimientos_invalidos = partidas_limpio['movimientos'] < 0

# Sustituimos solo los movimientos negativos por nulos y conservamos las partidas.
partidas_limpio.loc[movimientos_invalidos,'movimientos'] = float('nan')

print("Movimientos negativos:", (partidas_limpio['movimientos'] < 0).sum())
print("Movimientos nulos:", partidas_limpio['movimientos'].isna().sum())

Movimientos negativos: 0
Movimientos nulos: 32


### Limpieza de `partidas.csv`

- **Filas originales:** 61,716
- **`fecha`**: convertida a datetime (formatos mixtos, sin nulos resultantes).
- **`resultado`**: estandarizado a `1` (victoria) y `0` (derrota), unificando las variantes de texto.
- **`boosters_usados`**: detectados 1,854 nulos antes de deduplicar, aproximadamente el 3%. No se observó una concentración clara por nivel. Se conservan como desconocidos, sin sustituirlos por cero. Tras eliminar duplicados, quedan 1,842 nulos.
- **Duplicados exactos**: eliminadas 307 filas (`partida_id` repetido con fila idéntica). Sin identificadores repetidos tras la limpieza.
- **`movimientos`**: sustituidos 32 valores negativos por nulos, conservando las partidas para otros análisis. Se mantienen 13 valores iguales a cero (11 victorias y 2 derrotas), cuyo significado queda pendiente de aclarar.
- **Filas finales:** 61,409


### Limpieza de `compras.csv`

In [32]:
# Creamos la copia
compras_limpio = dfs['compras'].copy()

compras_antes = len(compras_limpio)

print(compras_limpio.dtypes)

compra_id      int64
user_id        int64
producto_id    int64
fecha            str
importe          str
moneda           str
plataforma       str
reembolso      int64
dtype: object


In [33]:
# Convertimos la fecha a datetime
compras_limpio['fecha'] = pd.to_datetime(compras_limpio['fecha'],format='mixed',dayfirst=True,errors='coerce')

print("Tipo ede fecha",compras_limpio['fecha'].dtype)
print("Fechas nulas o no reconocidas",compras_limpio['fecha'].isna().sum())

Tipo ede fecha datetime64[us]
Fechas nulas o no reconocidas 0


In [34]:
# Estandarizamos la plataforma, aplicando el diccionario existente
compras_limpio['plataforma'] = compras_limpio['plataforma'].str.strip().str.lower().map(mapa_plataforma)

# Comprobacion
print("Plataformas normalizadas")
print(compras_limpio['plataforma'].value_counts(dropna=False))

Plataformas normalizadas
plataforma
Android    2372
iOS        2186
Name: count, dtype: int64


In [35]:
# Revisamos las formas de la monedas
print("Monedas antes de normalizar:")
print(compras_limpio['moneda'].value_counts(dropna=False))

Monedas antes de normalizar:
moneda
EUR    3155
eur     921
€       470
MXN      12
Name: count, dtype: int64


In [36]:
# Unificamos las formas de escribir euros
mapa_moneda = {
    'eur': 'EUR',
    '€': 'EUR',
    'mxn': 'MXN'   
}

compras_limpio['moneda'] = compras_limpio['moneda'].str.lower().str.strip().map(mapa_moneda)

In [37]:
# Identificamos motivos de exclusión
es_reembolso = compras_limpio['reembolso'] == 1
otra_moneda = compras_limpio['moneda'] != 'EUR'

# COntamos cada problema y su coincidencia
print("Reembolsos",es_reembolso.sum())
print("Compras en otra moneda",otra_moneda.sum())
print("Ambos problemas",(es_reembolso & otra_moneda).sum())

# Contamos cada exclusion
print("Total filas a excluir",(es_reembolso | otra_moneda).sum())

# Conservamos únicamente operaciones no reembolsadas y en euros.
compras_limpio = compras_limpio.loc[(compras_limpio['reembolso']==0) & (compras_limpio['moneda']=='EUR')]

Reembolsos 67
Compras en otra moneda 12
Ambos problemas 0
Total filas a excluir 79


In [38]:
# Limpiamos importes con formato europeo, por ejemplo: "1.263,71 €".
# Quitamos el euro, los puntos de miles y los espacios de los extremos.
# Después cambiamos la coma decimal por un punto y convertimos a float.
compras_limpio['importe'] = (compras_limpio['importe']
                            .str.replace('€','',regex=False)
                            .str.replace('.', '', regex=False)
                            .str.replace(',', '.', regex=False)
                            .str.strip()
                            .astype(float))

In [39]:
# Comprobamos el tamaño final, el tipo y algunos importes.
print("\nVentas en euros restantes:", len(compras_limpio))
print("Tipo de importe:", compras_limpio['importe'].dtype)
print(compras_limpio['importe'].head())


Ventas en euros restantes: 4479
Tipo de importe: float64
0    4.99
1    0.99
2    0.99
3    1.99
4    2.49
Name: importe, dtype: float64


In [40]:
# Comprobamos que solo quedan operaciones no reembolsadas y en euros
print("Reembolsos restantes:", (compras_limpio['reembolso'] != 0).sum())
print("Compras fuera de EUR:", (compras_limpio['moneda'] != 'EUR').sum())

# Reviamos si hay importes ausentes, negativos o iguales a 0
print("\nImportes nulos:", compras_limpio['importe'].isna().sum())
print("Importes negativos:", (compras_limpio['importe']<0).sum())
print("Importes iguales a 0:", (compras_limpio['importe']==0).sum())

# Confirmamos que cada compra sigue teniendo un identificador único.
print(
    "\nIdentificadores de compra repetidos:",
    compras_limpio.duplicated(subset=['compra_id']).sum()
)

Reembolsos restantes: 0
Compras fuera de EUR: 0

Importes nulos: 0
Importes negativos: 0
Importes iguales a 0: 0

Identificadores de compra repetidos: 0


In [41]:
# Resumimos los importes para ver su distribución y sus extremos.
print("\nResumen de importes:")
print(compras_limpio['importe'].describe())


Resumen de importes:
count    4479.000000
mean        2.419895
std         3.147625
min         0.990000
25%         0.990000
50%         1.990000
75%         2.490000
max        49.990000
Name: importe, dtype: float64


### Limpieza de `compras.csv`

- **Filas originales:** 4,558
- **`fecha`**: convertida a datetime (formatos mixtos, sin nulos resultantes).
- **`plataforma`**: estandarizada a `Android` / `iOS`, reutilizando el diccionario de usuarios y sesiones, sin nulos resultantes.
- **`moneda`**: unificadas las variantes `EUR`, `eur` y `€` a `EUR`; mantenido `MXN` como divisa distinta.
- **Reembolsos y otras monedas**: excluidos 67 reembolsos y 12 compras en MXN, siguiendo el criterio del caso. No hay coincidencias entre ambos grupos: 79 filas excluidas en total.
- **`importe`**: eliminado el símbolo €, normalizados los separadores y convertido a `float64`. Sin nulos, negativos ni ceros. Importes entre 0.99 € y 49.99 €.
- **Duplicados**: sin repeticiones de `compra_id`.
- **Filas finales:** 4,479 ventas no reembolsadas en euros.

### Limpieza de `productos_iap.csv`

In [42]:
# Creamos la copia
productos_iap_limpio = dfs['productos_iap'].copy()

print(productos_iap_limpio.dtypes)

producto_id      int64
nombre             str
tipo               str
precio_base        str
monedas        float64
dtype: object


In [43]:
# Revisamos los tipos antes y cuantos productos tiene cada tipo
print("Tipos antes de normalizar:")
print(productos_iap_limpio['tipo'].value_counts(dropna=False))

Tipos antes de normalizar:
tipo
monedas            8
Monedas            3
suscripcion        2
quitar_anuncios    1
Name: count, dtype: int64


In [44]:
# Convertimos a minusculas quitamos espacios extremos 
productos_iap_limpio['tipo'] = productos_iap_limpio['tipo'].str.lower().str.strip()

# Categorias resultantes
print("Tipos despues de normalizar:")
print(productos_iap_limpio['tipo'].value_counts(dropna=False))

Tipos despues de normalizar:
tipo
monedas            11
suscripcion         2
quitar_anuncios     1
Name: count, dtype: int64


In [45]:
# Quitamos el símbolo €, los puntos de miles y los espacios exteriores.
productos_iap_limpio['precio_base'] = (
    productos_iap_limpio['precio_base']
    .str.replace('€','',regex=False)
    .str.replace('.','',regex=False)
    .str.replace(',','.',regex=False)
    .str.strip()
    .astype(float)
)

# Como solo hay 14 productos, podemos revisar todos sus precios.
print(productos_iap_limpio[['nombre', 'precio_base']])

                 nombre  precio_base
0        Pack Monedas S         0.99
1        Pack Monedas M         2.99
2        Pack Monedas L         4.99
3       Pack Monedas XL         9.99
4      Pack Monedas XXL        19.99
5     Pack Monedas Mega        49.99
6       Quitar Anuncios         3.99
7        Pase Temporada         7.99
8   Pase Temporada Plus        14.99
9            Pack Vidas         1.99
10          Pack Inicio         1.99
11         Pack Booster         2.49
12         Cofre Diario         0.99
13          Mega Oferta        29.99


In [46]:
# Comprobamos el tipo y si hay precios ausentes o no positivos.
print("\nTipo de precio:", productos_iap_limpio['precio_base'].dtype)
print("Precios nulos:", productos_iap_limpio['precio_base'].isna().sum())
print("Precios cero o negativos:", (productos_iap_limpio['precio_base'] <= 0).sum())


Tipo de precio: float64
Precios nulos: 0
Precios cero o negativos: 0


In [47]:
# Buscamos los productos cuya cantidad en monedas esta vacia
print("Productos con monedas nulas")
print(productos_iap_limpio.loc[productos_iap_limpio['monedas'].isna(),
      ['nombre','tipo','monedas']])

Productos con monedas nulas
                nombre             tipo  monedas
6      Quitar Anuncios  quitar_anuncios      NaN
7       Pase Temporada      suscripcion      NaN
8  Pase Temporada Plus      suscripcion      NaN


In [48]:
# Quitamos espacios al principio y al final de los nombres.
productos_iap_limpio['nombre'] = productos_iap_limpio['nombre'].str.strip()

print("Productos finales",len(productos_iap_limpio))

# Verificamos que ningún identificador esté vacío o repetido.
print("Identificadores nulos:",productos_iap_limpio['producto_id'].isna().sum())
print("Identificadores repetidos:",productos_iap_limpio.duplicated(subset=['producto_id']).sum())

Productos finales 14
Identificadores nulos: 0
Identificadores repetidos: 0


### Limpieza de `productos_iap.csv`

- **Filas originales:** 14
- **`tipo`**: normalizado a minúsculas y sin espacios exteriores. Quedan 3 categorías: `monedas` (11), `suscripcion` (2) y `quitar_anuncios` (1).
- **`precio_base`**: eliminado el símbolo €, normalizados los separadores y convertido a `float64`. Sin nulos, ceros ni negativos; precios entre 0.99 € y 49.99 €.
- **`monedas`**: conservados 3 nulos correspondientes a Quitar Anuncios y los dos pases de temporada. Se interpretan como «no aplica», según el criterio documentado en la exploración.
- **`nombre`**: eliminados los espacios exteriores, conservando las mayúsculas originales.
- **`producto_id`**: sin nulos ni identificadores repetidos.
- **Filas finales:** 14

### Limpieza de `ingresos_ads.csv`

In [49]:
# Creamos copia
ingresos_ads_limpio = dfs['ingresos_ads'].copy()

print(ingresos_ads_limpio.dtypes)

id           int64
user_id      int64
fecha          str
tipo_ad        str
red_ad         str
ingreso    float64
dtype: object


In [50]:
# Transformamos las fechas
ingresos_ads_limpio['fecha'] = pd.to_datetime(ingresos_ads_limpio['fecha'],
                                              format='mixed',
                                              dayfirst=True,
                                              errors='coerce')

In [51]:
# Comprobamos el tipo resultante y las fechas que han quedado vacías.
print("Tipo de fecha:", ingresos_ads_limpio['fecha'].dtype)
print("Fechas nulas o no reconocidas:", ingresos_ads_limpio['fecha'].isna().sum())

Tipo de fecha: datetime64[us]
Fechas nulas o no reconocidas: 0


In [52]:
# Contamos las distintas formas de escribir el tipo de anuncio.
print("Tipos de anuncio antes de normalizar:")
print(ingresos_ads_limpio['tipo_ad'].value_counts(dropna=False))

Tipos de anuncio antes de normalizar:
tipo_ad
rewarded        4047
recompensado    3993
Rewarded        3925
interstitial    3094
Interstitial    2946
banner          1007
Banner           987
Name: count, dtype: int64


In [53]:
# Definimos 3 tipos de anuncio 
mapa_tipo_ad = {
    'rewarded': 'rewarded',
    'recompensado': 'rewarded',
    'interstitial': 'interstitial',
    'banner': 'banner'
}

ingresos_ads_limpio['tipo_ad'] = (ingresos_ads_limpio['tipo_ad']
                                  .str.lower()
                                  .str.strip()
                                  .map(mapa_tipo_ad))

# Mostramos los recuentos incluyendo nulos para detectar valores sin mapear.
print("Tipos de anuncio normalizados:")
print(ingresos_ads_limpio['tipo_ad'].value_counts(dropna=False))


Tipos de anuncio normalizados:
tipo_ad
rewarded        11965
interstitial     6040
banner           1994
Name: count, dtype: int64


In [54]:
# Revisamos también los nombres de las redes publicitarias.
print("\nRedes publicitarias antes de normalizar")
print(ingresos_ads_limpio['red_ad'].value_counts(dropna=False))


Redes publicitarias antes de normalizar
red_ad
admob         6121
AdMob         6031
ironsource    3957
ironSource    3890
Name: count, dtype: int64


In [55]:
# Solo cambiamos mayusculas 
ingresos_ads_limpio['red_ad'] = (
    ingresos_ads_limpio['red_ad'].str.lower().str.strip()
)

print("Redes publicitarias normalizadas:")
print(ingresos_ads_limpio['red_ad'].value_counts(dropna=False))

Redes publicitarias normalizadas:
red_ad
admob         12152
ironsource     7847
Name: count, dtype: int64


In [62]:
# Resumimos los ingresos informados: mínimo, máximo, media y cuartiles.
print("Resumen ingresos:")
print(ingresos_ads_limpio['ingreso'].describe())

# Contamos por separado los datos ausentes, los ceros y los negativos.
print("\nIngresos nulos:",ingresos_ads_limpio['ingreso'].isnull().sum())
print("Ingresos iguales a 0:",(ingresos_ads_limpio['ingreso']==0).sum())
print("Ingresos negativos:",(ingresos_ads_limpio['ingreso']<0).sum())

Resumen ingresos:
count    19399.000000
mean         0.206093
std          0.178746
min          0.000200
25%          0.074650
50%          0.155900
75%          0.284550
max          1.872400
Name: ingreso, dtype: float64

Ingresos nulos: 600
Ingresos iguales a 0: 0
Ingresos negativos: 0


In [68]:
# Creamos una tabla auxiliar para ver los ingresos ausentes
revision_ingresos = ingresos_ads_limpio.assign(
    ingreso_nulo = ingresos_ads_limpio['ingreso'].isna()
    )

# Agrupamos por red y anuncio
resumen_nulos = (
    revision_ingresos.groupby(['red_ad', 'tipo_ad'])
    .agg(
        total_anuncios=('ingreso_nulo', 'size'),
        ingresos_nulos=('ingreso_nulo', 'sum')
    ) )

resumen_nulos['porcentaje_nulos'] = (
    resumen_nulos['ingresos_nulos']
    / resumen_nulos['total_anuncios']
    * 100
)

print(
    resumen_nulos
    .sort_values('porcentaje_nulos', ascending=False)
    .round(2)
)

                         total_anuncios  ingresos_nulos  porcentaje_nulos
red_ad     tipo_ad                                                       
ironsource banner                   793              30              3.78
admob      interstitial            3672             117              3.19
ironsource rewarded                4686             145              3.09
admob      banner                  1201              36              3.00
           rewarded                7279             212              2.91
ironsource interstitial            2368              60              2.53


In [ ]:
print("Anuncios finales:", len(ingresos_ads_limpio))

# Revisamos que cada anuncio tenga un identificador informado y único.
print("Identificadores nulos:", ingresos_ads_limpio['id'].isna().sum())
print("Identificadores repetidos:",
    ingresos_ads_limpio.duplicated(subset=['id']).sum()
)

Anuncios finales: 19999
Identificadores nulos: 0
Identificadores repetidos: 0


### Limpieza de `ingresos_ads.csv`

- **Filas originales:** 19,999
- **`fecha`**: convertida a datetime (formatos mixtos, sin nulos resultantes).
- **`tipo_ad`**: normalizado a 3 categorías: `rewarded`, `interstitial` y `banner`, unificando mayúsculas y traduciendo `recompensado` a `rewarded`.
- **`red_ad`**: normalizada a minúsculas y sin espacios exteriores. Quedan 2 redes: `admob` e `ironsource`.
- **`ingreso`**: ya era numérico. Sin ceros ni negativos entre los valores informados. Se conservan 600 nulos (aproximadamente el 3%) como ingresos desconocidos, manteniendo los anuncios.
- **Distribución de nulos**: porcentajes entre el 2.53% y el 3.78% por combinación de red y tipo de anuncio, sin una concentración marcada. Esto no demuestra que sean aleatorios.
- **Limitación para los KPIs**: la suma reflejará el ingreso registrado, sin incluir los importes desconocidos.
- **`id`**: sin nulos ni identificadores repetidos.
- **Filas finales:** 19,999

### Limpieza de `gasto_ua.csv`

In [73]:
gasto_ua_limpio = dfs['gasto_ua'].copy()

print(gasto_ua_limpio.dtypes)

canal                str
fecha                str
gasto                str
instalaciones      int64
impresiones        int64
clics            float64
dtype: object


In [74]:
# Fechas a formato datetime
gasto_ua_limpio['fecha'] = pd.to_datetime(gasto_ua_limpio['fecha'],
                        format='mixed',
                        dayfirst=True,
                        errors='coerce'            
                        )


# Comprobamos el tipo resultante y las fechas ausentes.
print("Tipo de fecha:", gasto_ua_limpio['fecha'].dtype)
print("Fechas nulas o no reconocidas:", gasto_ua_limpio['fecha'].isna().sum())

Tipo de fecha: datetime64[us]
Fechas nulas o no reconocidas: 0


In [75]:
# Unificamos los canales con los mismo nombres usados en usuarios
# Permite una posterior relaciones de los gastos de los jugadores
gasto_ua_limpio['canal'] = gasto_ua_limpio['canal'].str.lower().str.strip().map(mapa)

# Analisis canales
print("Canales normalizados")
print(gasto_ua_limpio['canal'].value_counts(dropna=False))

Canales normalizados
canal
apple_search    50
meta            50
organico        50
google          50
tiktok          50
coinrush        30
Name: count, dtype: int64


In [76]:
# Estandarizacion de gasto a numero, quitando simbolos y separadores de miles
gasto_ua_limpio['gasto'] = (
    gasto_ua_limpio['gasto']
    .str.replace('€','',regex=False)
    .str.replace('.','',regex=False)
    .str.replace(',','.',regex=False)
    .str.strip()
    .astype(float)
)

# Reviamos tipo y distribucion
print("Tipo de gasto",gasto_ua_limpio['gasto'].dtype)
print("\nResumen del gasto:")
print(gasto_ua_limpio['gasto'].describe())

# Contamos posibles valores que requieran revisión.
print("\nGastos nulos:", gasto_ua_limpio['gasto'].isna().sum())
print("Gastos negativos:", (gasto_ua_limpio['gasto'] < 0).sum())
print("Gastos iguales a cero:", (gasto_ua_limpio['gasto'] == 0).sum())

Tipo de gasto float64

Resumen del gasto:
count     280.000000
mean       32.357179
std       119.217326
min         0.000000
25%         4.875000
50%        15.440000
75%        29.330000
max      1263.710000
Name: gasto, dtype: float64

Gastos nulos: 0
Gastos negativos: 0
Gastos iguales a cero: 50
